<center><h2><strong><font color="blue"> Advanced Programming for Data Science (APDS)</font></strong></h2></center>

<center><img alt="" src="images/covers/taudata-cover.jpg"/></center>

<center><h2><strong><font color="blue">APDS-12: Multi-Processing in Python</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

# Review Basic Python

* Try-Except
* Function
* Thread Programming in Python - GIL

# Concurrency versus Parallelism

* While conceptually similar, concurrency typically refers to managing multiple threads, whereas parallelism involves the simultaneous execution of tasks across multiple processors.

> A system is said to be concurrent if it can support two or more actions in progress at the same time. A system is said to be parallel if it can support two or more actions executing simultaneously. The key concept and difference between these definitions is the phrase “in progress.”

<center><img alt="" src="images/APDS/concurrent-vs-parallel.jpg" style="height: 250px;"/></center>

* image source: https://medium.com/@sanju.skm/parallel-programming-vs-concurrent-programming-f993d3f9ceea

# Program, Process, & Threads

* **A Program** is an executable file containing a set of instructions and passively stored on disk. One program can have multiple processes. For example, the Chrome browser creates a different process for every single tab.
* **A Process** means a program is in execution. When a program is loaded into the memory and becomes active, the program becomes a process. The process requires some essential resources such as registers, program counter, and stack.
* **A Thread** is the smallest unit of execution within a process.

<center><img alt="" src="images/APDS/program-process-thread.png" style="height: 400px;"/></center>
image source: https://bytebytego.com/guides/what-is-the-difference-between-process-and-thread/

* The program contains a set of instructions.
* The program is loaded into memory. It becomes one or more running processes.
* When a process starts, it is assigned memory and resources. A process can have one or more threads. For example, in the Microsoft Word app, a thread might be responsible for spelling checking and the other thread for inserting text into the doc.

# Serial VS Threaded VS multiprocess in Python

<center><img alt="" src="images/APDS/Serial-Thread-multiprocess-in-Python.png" style="height: 250px;"/></center>
image source: https://medium.com/data-science/multithreading-and-multiprocessing-in-10-minutes-20d9b3c6a867

# Usecase in Data Science

<center><img alt="" src="images/APDS/Thread-vs-parallel-programming-usecase-in-dataScience.jpg" style="height: 400px;"/></center>
image source: AI Generated

# Simplest Implementation

In [ ]:
from multiprocessing import Process
import os

def worker():
    print(f"Worker process ID: {os.getpid()}")

if __name__ == "__main__":
    print(f"Main process ID: {os.getpid()}")
    p = Process(target=worker)
    p.start()
    p.join()

# Running Multiple Processes in Parallel

In [ ]:
from multiprocessing import Process
import os

def worker(n):
    print(f"Worker {n}, PID: {os.getpid()}")

if __name__ == "__main__":
    processes = []

    for i in range(4):
        p = Process(target=worker, args=(i,))
        processes.append(p)
        p.start()

    for p in processes:
        p.join()

# The Pool Class

* Tasks are processed based on the input sequence.

<img alt="" src="images/pool_mp_python.png" />

### It is worth evaluating why the following example might not necessarily outperform serial execution.

In [ ]:
import multiprocessing as mp

def f(x):
    return x*x

if __name__ == '__main__':
    print('Number of currently available processor = ', mp.cpu_count())
    input_ = [1, 2, 3, 4, 5, 7, 9, 10]
    print('input = ', input_)
    with mp.Pool(5) as p:
        print(p.map(f, input_))

Number of currently available processor =  16


# CPU-Bound Example (Why Multiprocessing Matters)

In [ ]:
from multiprocessing import Pool
import math

def cpu_task(n):
    return sum(math.sqrt(i) for i in range(n))

if __name__ == "__main__":
    data = [10**7] * 4

    with Pool(4) as pool:
        results = pool.map(cpu_task, data)

    print(results)

# Practical Implementation of the Pool Class

In [ ]:
import multiprocessing
import numpy as np
import time

def pungsi(N):
    s = 0.0
    for i in range(1,N):
        s += np.log(i)
    return s

if __name__ == '__main__':
    inputs = [10**6] * 20
    print('Sequential Execution Performance:')
    mulai =  time.time()
    outputs = [pungsi(x) for x in inputs]
    akhir  = time.time()
    print("Mean Output: {}".format(np.mean(outputs)))
    print("Serial Execution Time: {}".format(akhir-mulai))
    
    print('Parallel Execution Performance:')
    mulai =  time.time()
    pool = multiprocessing.Pool()
    pool = multiprocessing.Pool(processes=8)
    outputs = pool.map(pungsi, inputs)
    akhir  = time.time()
    #print("Input: {}".format(inputs))
    print("Mean Output: {}".format(np.mean(outputs)))
    print("Parallel Execution Time: {}".format(akhir-mulai))

# Asynchronous Mapping (map_async)

<img alt="" src="images/sync_vs_async.png" />

In [ ]:
import multiprocessing as mp

def square(x):
    return x * x

if __name__ == '__main__':
    inputs = [0,1,2,3,4,5,6,7,8]
    
    print('Synchronous Parallel Processing')
    pool = mp.Pool()
    outputs = pool.map(square, inputs)
    print("Input: {}".format(inputs))
    print("Output: {} \n".format(outputs))
    pool.close(); del pool
    
    print('Asynchronous Parallel Processing')
    pool = mp.Pool()
    outputs_async = pool.map_async(square, inputs)
    outputs = outputs_async.get()
    print("Input: {}".format(inputs))
    print("Output: {}".format(outputs))

# Manual Process Allocation

Specific functions may be assigned to individual processors manually if granular control is required.

In [ ]:
import multiprocessing
import os
import time
import threading

class ProsesA(multiprocessing.Process):
    def __init__(self, id):
        super(ProsesA, self).__init__()
        self.id = id

    def run(self):
        time.sleep(1)
        print("PID: %s, Process ID: %s, Process Name: %s, Thread Name: %s" % (
        os.getpid(), self.id,
        multiprocessing.current_process().name,
        threading.current_thread().name))
        
class ProsesB(multiprocessing.Process):
    def __init__(self, id):
        super(ProsesB, self).__init__()
        self.id = id

    def run(self):
        time.sleep(1)
        print("PID: %s, Process ID: %s, Process Name: %s, Thread Name: %s" % (
        os.getpid(), self.id,
        multiprocessing.current_process().name,
        threading.current_thread().name))

if __name__ == '__main__':
    p1 = ProsesA(0)
    p1.start()
    p2 = ProsesB(1)
    p2.start()
    p1.join(); p2.join()

# Parallel Execution for Multivariate Functions: StarMap

In Python’s multiprocessing module, starmap is a method of a process pool that applies a function to multiple sets of arguments in parallel. Unlike map, which passes a single iterable of values to a function, starmap expects an iterable of argument tuples and unpacks each tuple when calling the target function.

This makes starmap particularly useful when the worker function requires multiple parameters, as it allows clean and readable parallel execution without manual argument packing or wrapper functions.

<center><img alt="" src="images/APDS/starMap-in-Python.jpg" style="height: 400px;"/></center>

In [ ]:
import multiprocessing as mp

def f_sum(a, b):
    return a + b

if __name__ == '__main__':
    process_pool = mp.Pool(4)
    data = [(1, 1), (2, 1), (3, 1), (6, 9)]
    output = process_pool.starmap(f_sum, data)
    print("input = ", data)
    print("output = ", output)

# End of Module

* Next week: **Introduction to GPU Programming**

<hr>
<img alt="" src="images/meme-cartoon/meme-multithread-vs-parallel.jpg"/>